# EchoNest ReAct agent with personas (Assignment C4)

EECE 503P/798S, Agentic Systems. A ReAct agent for **EchoNest**, a fictional digital-memory studio, built by hand in
**LangGraph** (no prebuilt executor), with **three personas** defined by prompt traits *and* LLM settings, a dedicated
**instructions block**, and **logged experiments** comparing personas, LLM settings, models and prompt techniques.

The code lives in the `echonest_agent/` package; this notebook imports it, explains it, and shows the results.

| Section | PDF item |
|---|---|
| [1. Use case](#1.-Use-case) | 2.1 use case |
| [2. The ReAct loop](#2.-The-ReAct-loop-(LangGraph,-written-by-hand)) | 1, 2.2 ReAct logic |
| [3. Tools](#3.-Tools) | 2.2 tool descriptions in the prompt |
| [4. Personas](#4.-Personas:-prompt-traits-and-matching-LLM-settings) | 2.3 personas |
| [5. Instructions block](#5.-The-instructions-block) | 2.4 instructions |
| [6. Demo](#6.-Demo:-one-question-per-persona) | 1 working assistant |
| [7. Experiments](#7.-Experiments) | 2.3 configurations and prompt engineering, logged and compared |
| [8. Reflection](#8.-Reflection) | 2.5 reflection questions |
| [9. Limitations and how to run](#9.-Limitations-and-how-to-run) | |

**`RUN_LIVE = False`** (below) runs the whole notebook offline from stored results: no API key, no quota. Set it to
`True` to send real requests in section 6 (about 10 requests to the model chosen in `.env`).

In [1]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from echonest_agent import graph, personas, prompts, tools
from echonest_agent.react_parser import parse_react

RUN_LIVE = False          # True: section 6 sends real requests (model from .env); False: stored results only
pd.set_option("display.max_colwidth", 80)
SUMMARY = ROOT / "experiments" / "summary"


def show(name, cols=("config_id", "runs", "success", "tool_precision", "format_errors", "steps", "requests",
                     "latency_s", "words", "distinct_2")):
    """One summary table from experiments/summary/<name>.csv (written by scripts/summarize_experiments.py)."""
    df = pd.read_csv(SUMMARY / f"{name}.csv")
    cols = [c for c in cols if c in df] + (["distinct_2_rep"] if df.get("distinct_2_rep") is not None
                                            and df["distinct_2_rep"].notna().any() else [])
    display(df[cols].set_index("config_id"))

## 1. Use case

The scenario is reused from the previous assignment (PDF 2.1 allows it). What is new is *how* the agent works: it has
no business facts in its prompt, so it must look them up with tools inside a ReAct loop.

In [2]:
display(Markdown((ROOT / "docs" / "use_case.md").read_text(encoding="utf-8")))

# Use case: the EchoNest ReAct agent

Assignment C4 lets us reuse the previous assignment's scenario (PDF §2.1), so the agent still works for **EchoNest**,
a fictional digital-memory studio. EchoNest turns people's photos, voice recordings and written memories into a
private, shareable story page. It sells five packages: Memory Capsule $79, Event Story $149, Life Timeline $299,
Legacy Edition $499, and Corporate Story, which has no fixed price. What's new is **how** the agent works: it now
**reasons and acts in a ReAct loop**. It has no business facts in its prompt, so it has to look them up with tools.

## Agent goal

Help a website visitor understand EchoNest's services and choose one, with every fact taken from EchoNest's own
documents. Then, depending on the conversation:
- capture the visitor as a **lead** once they want a project;
- **log** what the documents can't answer (or a service EchoNest doesn't offer), so the team sees the demand;
- **log feedback** (complaints, praise).

Three personas share this goal but have **different tasks**, as PDF §2.3 asks. They are designed in Stage 5 of the
roadmap:

| Persona | Task |
|---|---|
| Milo, the Warm Concierge | first contact, general questions, turning interest into a lead |
| Nora, the Package Specialist | exact prices, package comparisons, payment and policy details |
| Sami, the Story Planner | helping the customer plan their story (which memories, what structure) within a package's limits |

## Target audience

- **Families:** family histories, anniversaries, birthdays and reunions.
- **Individuals:** graduations, relationships, travels and personal milestones.
- **Organisations:** founder stories, company milestones and employee anniversaries.

They are website visitors with no technical background, writing in English, Arabic or French. The documents
themselves are in English.

## Tools it can use

Defined in [`echonest_agent/tools.py`](../echonest_agent/tools.py). Their descriptions are generated from the same
registry into the system prompt. Every tool returns a short text **observation**.

| Tool | What it does | Typical trigger |
|---|---|---|
| `search_business_docs(query)` | Keyword (BM25) search over the business PDF and summary: up to 3 passages with their source section, plus the query words that appear in **no** document | any factual question about EchoNest |
| `get_package(name)` | The exact price, contents, payment terms and timeline of one package, read from the PDF (nothing typed by hand) | "How much is the Legacy Edition?", "What does a Memory Capsule include?" |
| `record_customer_interest(email, name, message)` | Saves a lead to `logs/leads.jsonl` (an invalid email is refused, and nothing is saved) | the customer wants a project and has given a name and email |
| `record_feedback(question)` | Saves an unanswered question, a request for a service EchoNest doesn't offer, or feedback, to `logs/feedback.jsonl` | a search shows no answer; "Can you film our wedding?"; "The upload link is broken" |

## Constraints and limitations

**Rules the agent must follow:**
- **Grounded answers only.** Facts about EchoNest come from tool observations. The agent never invents prices,
  dates, people, discounts or policies. The Corporate Story has no fixed price, so the agent never gives a number.
- **What's missing is logged, not guessed.** The documents deliberately leave things out: gift cards, discounts,
  albums, rush orders, a "Platinum" tier. These must go to `record_feedback`. So must the four services EchoNest
  explicitly doesn't offer: film restoration, physical photo restoration, videography, recovery of deleted files.
- **No transactions.** The agent can't book, take payments or accept files; materials go through a secure upload
  link after booking. It only collects a name, an email and a one-line project description.
- **Leads need a valid email**, and the agent must never claim it saved something unless a tool call did.

**Technical limitations:**
- **Keyword search,** not semantic search. A question phrased very differently from the documents may need a second
  search with other words. The observation lists the words that matched nothing, to help the agent decide.
- **Free-tier LLMs.** Gemini 3.5 Flash-Lite allows 500 requests a day, and each ReAct step is one request. It
  ignores temperature and top-p, and rejects top-k and the penalties. That's why the settings experiments also run
  on Gemma 4 and a local model (see [stage0_findings.md](stage0_findings.md)).
- **Memory is the chat history only:** nothing is remembered between sessions. The logs are local files (lost on a
  Hugging Face Space restart).
- **The model is configuration.** Switching to another model or provider changes `.env` only (see the README). How
  well the agent follows the ReAct format depends on the model, and the experiments measure this.


## 2. The ReAct loop (LangGraph, written by hand)

The loop is a `StateGraph` in `echonest_agent/graph.py` with five nodes and one **conditional edge**,
`route_after_reason`, which **detects whether the model suggested an action** (PDF 2.2). No `create_react_agent`,
`ToolNode`, `tools_condition`, `AgentExecutor` or native tool calling (`bind_tools`) is used;
`tests/test_graph.py::test_no_prebuilt_agent_is_used` checks the source.

- **reason**: one LLM call (system prompt + chat history + this turn's scratchpad), with `stop=["Observation:"]` so the
  model cannot invent observations; then `parse_react()` reads the reply.
- **act**: runs the tool; its result goes back as `Observation: ...` under the step.
- **recover**: the reply broke the format: a correction is sent (at most 2 per turn).
- **finalize**: returns the Final Answer. **give_up**: too many steps: apologise and log the question.

The protocol is **text ReAct** (Thought / Action / Action Input / Final Answer), so it works with any chat model.

In [3]:
class NoLLM:                       # the graph's shape needs no model
    def complete(self, *args, **kwargs):
        raise RuntimeError("not used")


print(graph.ReActAgent(llm=NoLLM(), verbose=False).graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	act(act)
	recover(recover)
	finalize(finalize)
	give_up(give_up)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	act --> reason;
	reason -.-> act;
	reason -.-> finalize;
	reason -.-> give_up;
	reason -.-> recover;
	recover --> reason;
	finalize --> __end__;
	give_up --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**Detecting the action and parsing the final answer.** `parse_react()` turns a reply into an action, a final answer,
or an invalid step with a correction message. It accepts the PDF's own one-line example:

In [4]:
pdf_example = ('Thought: I need to look up location info. Action: lookup_location("Paris") '
               'Observation: Paris is the capital of France.')
step = parse_react(pdf_example, tools={"lookup_location": ["location"]})   # the PDF's tool, one argument
print(step.kind, step.tool, step.args, "| thought:", step.thought, "| warnings:", step.warnings)

final = parse_react("Thought: That helps. Now I can answer.\nAnswer: Paris is a great travel destination!")
print(final.kind, "|", final.answer)

bad = parse_react("Let me check that for you.")
print(bad.kind, "|", bad.error)

action lookup_location {'location': 'Paris'} | thought: I need to look up location info. | warnings: ["the reply contained its own 'Observation:'; everything from there was ignored"]
final | Paris is a great travel destination!
invalid | the reply has no 'Action:' or 'Final Answer:' line


## 3. Tools

The descriptions below are generated from the tool registry (`echonest_agent/tools.py`) and embedded in the system
prompt (PDF 2.2). Each tool returns a short text observation. Two only read the documents; two write to the logs.

In [5]:
print(tools.describe_tools())

### search_business_docs
- Purpose: Search EchoNest's business documents (services, prices, policies, team, materials, FAQ).
- Use it: Before stating any fact about EchoNest that you have not seen in an Observation in this chat.
- Input (a JSON object): query (required): a few keywords about what the customer asked
- Output: Up to 3 passages with their source, then the query words that appear in NO document. NO_MATCH if nothing matches.
- Example: Action: search_business_docs / Action Input: {"query": "Legacy Edition delivery time"}

### get_package
- Purpose: Exact price, contents, payment terms and timeline of one EchoNest package.
- Use it: When the customer asks about a specific package: Memory Capsule, Event Story, Life Timeline, Legacy Edition or Corporate Story.
- Input (a JSON object): name (required): the package name
- Output: The package's facts, or ERROR listing the valid package names.
- Example: Action: get_package / Action Input: {"name": "Event Story"}

### record_custo

In [6]:
print(tools.safe_execute("get_package", {"name": "Legacy Edition"}))
print()
print(tools.safe_execute("search_business_docs", {"query": "gift cards"}))

Legacy Edition
Price: $499
Includes: our premium service for preserving someone's life story. The customer provides photographs, documents and memories, and we include up to 3 recorded interview sessions, held online.
Payment: a 50% deposit is paid at booking and the remaining balance is paid on delivery.
Timeline: Typical projects take 5 to 10 business days from the moment all materials are uploaded. The Legacy Edition takes 3 to 4 weeks because it includes interviews.
(Source: about_business.pdf, sections 4, 5 and 10.)

NO_MATCH: nothing in the EchoNest documents matches 'gift cards'.


## 4. Personas: prompt traits and matching LLM settings

Three personas with **different tasks** (PDF 2.3). Each is defined by prompt traits (tone, vocabulary, response
style, values) and by settings that match them, each with a one-line rationale (`echonest_agent/personas.py`). Length
and tone live only in the personas; rules live only in the instructions, so the two cannot conflict.

In [7]:
rows = [{"persona": f"{p.name}, {p.role}", "task": p.task_focus, **p.traits} for p in personas.PERSONAS.values()]
display(pd.DataFrame(rows).set_index("persona"))

settings = pd.DataFrame({p.name: p.settings for p in personas.PERSONAS.values()})
display(settings)
for p in personas.PERSONAS.values():
    print(f"\n{p.name}:")
    for name, why in p.rationale.items():
        print(f"  {name:<18} {why}")

,task,tone,vocabulary,response_style,values
persona,,,,,
"Milo, the Memory Concierge","first contact: welcome people, answer their general questions, and turn inte...","warm, curious and reassuring, like a friendly host; slightly conversational,...",everyday words with no jargon; you use the customer's name once you know it.,"two to four short, natural sentences (under about 100 words), with a list on...",people should feel heard: you show genuine interest in whose memories these ...
"Nora, the Package Specialist","exact prices, package comparisons, deposits and policies, so the customer ca...","calm, precise and matter-of-fact; polite but not chatty.","exact figures and package names, plain business terms (deposit, balance, rev...","lead with the number or the direct answer, then at most three short bullet p...","accuracy over warmth, and transparency about costs: what is included, what i..."
"Sami, the Story Planner","planning the story itself: chapters, which photos and recordings to gather, ...","enthusiastic, imaginative and encouraging, like a creative partner.","vivid storytelling words (chapters, scenes, moments, voices, turning points).","a one-sentence opening, then a numbered plan or list of ideas with concrete ...","every life story deserves a thoughtful shape, and a good plan fits what the ..."


,Milo,Nora,Sami
temperature,0.7,0.1,1.1
top_p,0.9,0.5,0.95
top_k,40,5,64
frequency_penalty,0.2,0.0,0.4
presence_penalty,0.3,0.0,0.6
max_tokens,400,700,800
reasoning_effort,low,medium,minimal



Milo:
  temperature        0.7: natural, varied wording, so a host doesn't sound scripted, without drifting off-fact.
  top_p              0.9: keeps the long tail of odd words out while leaving room for a conversational voice.
  top_k              40: a middle value; enough choice for natural phrasing.
  frequency_penalty  0.2: a light push against repeating the same phrases across a chat.
  presence_penalty   0.3: mild encouragement to bring up new topics (the occasion, the people involved).
  max_tokens         400: short replies plus head-room for the Thought line and low thinking.
  reasoning_effort   low: general questions need little hidden reasoning; keeps replies fast.

Nora:
  temperature        0.1: nearly deterministic, so the same question gets the same precise answer.
  top_p              0.5: only the most likely wording; no creative paraphrase of numbers.
  top_k              5: a narrow choice of tokens for repeatable figures and names.
  frequency_penalty  0.0: repea

**Not every model honours every setting.** We measured it (`scripts/probe_settings.py`, `docs/settings_support.md`)
instead of trusting the docs. The persona card shows what each persona *intends* and what is actually *sent*:

In [8]:
print((ROOT / "docs" / "settings_support.md").read_text(encoding="utf-8").split("## Evidence")[0])

# Which LLM settings each model honours

Generated by `python scripts/probe_settings.py --report` from `config/capabilities/*.json`. Do not edit by hand: re-probe a model, then regenerate.

**honoured** = accepted and the effect was measured · **ignored** = accepted (HTTP 200) but no measurable effect · **rejected** = HTTP 400 for a valid value · raw requests and replies are in `docs/probe/`.

| Setting | `gemini-3.5-flash-lite` (gemini) | `gemma-4-26b-a4b-it` (gemini) | `gemini-3.5-flash-lite` (gemini_native) | `gemma-4-26b-a4b-it` (gemini_native) | `qwen3:4b-instruct` (ollama) | `qwen3:4b-instruct` (ollama_native) |
|---|---|---|---|---|---|---|
| `temperature` | ⚪ ignored | ✅ honoured | ⚪ ignored | ✅ honoured | ✅ honoured | ✅ honoured |
| `top_p` | ⚪ ignored | ✅ honoured | ⚪ ignored | ✅ honoured | ✅ honoured | ✅ honoured |
| `top_k` | ❌ rejected | ❌ rejected | ⚪ ignored | ⚪ ignored | ⚪ ignored | ✅ honoured |
| `frequency_penalty` | ❌ rejected | ❌ rejected | ❌ rejected | ❌ rejected |

In [9]:
def llm_for(capability_file):
    caps = json.loads((ROOT / "config" / "capabilities" / capability_file).read_text(encoding="utf-8"))
    return SimpleNamespace(profile=SimpleNamespace(capabilities=caps),
                           backend=SimpleNamespace(can_express=lambda name, entry: True))


for label, file in [("gemini-3.5-flash-lite (the agent)", "gemini_native__gemini-3.5-flash-lite.json"),
                    ("qwen3:4b-instruct (local, Ollama native)", "ollama_native__qwen3_4b-instruct.json")]:
    llm = llm_for(file)
    table = {p.name: {name: (f"{intended}" if effective == intended else f"{intended} → {effective}")
                      for name, intended, effective, _ in p.card(llm)} for p in personas.PERSONAS.values()}
    print(label, "(intended → sent; — = not sent)")
    display(pd.DataFrame(table))

gemini-3.5-flash-lite (the agent) (intended → sent; — = not sent)


,Milo,Nora,Sami
temperature,0.7 → —,0.1 → —,1.1 → —
top_p,0.9 → —,0.5 → —,0.95 → —
top_k,40 → —,5 → —,64 → —
frequency_penalty,0.2 → —,0.0 → —,0.4 → —
presence_penalty,0.3 → —,0.0 → —,0.6 → —
max_tokens,400,700,800
reasoning_effort,low,medium,minimal


qwen3:4b-instruct (local, Ollama native) (intended → sent; — = not sent)


,Milo,Nora,Sami
temperature,0.7,0.1,1.1
top_p,0.9,0.5,0.95
top_k,40,5,64
frequency_penalty,0.2,0.0,0.4
presence_penalty,0.3,0.0,0.6
max_tokens,400,700,800
reasoning_effort,low,medium,minimal


## 5. The instructions block

Shared by every persona (PDF 2.4): **Inputs you receive**, **Rules** (one purpose and one sentence each), a **Worked
example** built from the real tool output, and the **Output format** (last: with it in the middle, Flash-Lite returned
empty replies for 5 of 6 identical requests; last, 1 of 6). `tests/test_personas.py` checks the headings, the
one-sentence rules, that the example parses with `parse_react()`, and that persona and instructions don't overlap.

In [10]:
print(prompts.build_instructions())

## Inputs you receive
- The conversation so far: the customer's earlier messages and your earlier Final Answers.
- The customer's latest message: the one you answer now.
- Observations: the result of each tool you call, on a line starting with "Observation:" under your step. This
  turn's steps and Observations are shown back to you, followed by a request for your next step.

## Rules
1. Before you state a fact about EchoNest (a price, package, timeline, person or policy), look it up with search_business_docs or get_package, unless an Observation in this conversation already contains it.
2. Never state a price, date, person or policy about EchoNest that does not come from an Observation in this conversation (amounts you calculate from its figures, such as a deposit or a difference, are fine).
3. The Corporate Story has no fixed price, so never give a number or a range for it.
4. If a search shows the documents do not answer the customer's question about EchoNest (NO_MATCH, or its key w

## 6. Demo: one question per persona

The same question to the three personas. With `RUN_LIVE = False` the stored traces of this question from the
Stage 8 persona experiment are shown (`experiments/results/p_flash.jsonl`, case F2); with `True` the agent answers
now. The Gradio app (`python app.py`)
is the working assistant, with a persona selector and the trace shown as collapsible steps
(`docs/screenshots/09`–`11`).

In [11]:
QUESTION = "How long does the Legacy Edition take?"


def print_trace(name, trace, answer):
    print(f"===== {name} =====")
    for t in trace:
        if t["node"] == "reason" and t.get("thought"):
            print("🧠", t["thought"])
        elif t["node"] == "act":
            obs = t["observation"]
            print("🔧", t["tool"], t["args"], "\n👁 ", obs[:160] + (" …" if len(obs) > 160 else ""))
        elif t["node"] == "recover":
            print("🛠  format correction:", t["error"])
    print("✅", answer, "\n")


if RUN_LIVE:
    from echonest_agent import agent
    for p in personas.PERSONAS.values():
        reply = agent.respond(QUESTION, persona_id=p.id)
        print_trace(p.name, reply.result.trace if reply.result else [], reply.answer)
else:
    stored = [json.loads(line) for line in (ROOT / "experiments/results/p_flash.jsonl").read_text().splitlines()]
    for row in stored:
        if row["case_id"] == "F2":
            print_trace(f"{row['persona'].capitalize()} ({row['run_id']})", row["trace"], row["answer"])

===== Milo (p_flash:milo-flash:F2:0) =====
🧠 I need to check the timeline for the Legacy Edition package, so I will look up its details.
🔧 get_package {'name': 'Legacy Edition'} 
👁  Legacy Edition
Price: $499
Includes: our premium service for preserving someone's life story. The customer provides photographs, documents and memories, and we  …
🛠  format correction: the reply has no 'Action:' or 'Final Answer:' line
🧠 I have the timeline for the Legacy Edition from the previous observation, so I can answer the customer directly and ask about their project.
✅ The Legacy Edition takes 3 to 4 weeks to complete because it includes recorded interviews. Is this for a special occasion or a particular person you have in mind? 

===== Nora (p_flash:nora-flash:F2:0) =====
🧠 I need to check the timeline for the Legacy Edition package, so I will use the get_package tool.
🔧 get_package {'name': 'Legacy Edition'} 
👁  Legacy Edition
Price: $499
Includes: our premium service for preserving someone's lif

## 7. Experiments

**Method** (`tests/eval_cases.json`, `echonest_agent/evaluation.py`, `scripts/run_experiments.py`): 34 frozen cases
in 12 categories (facts, prices, unanswerable, not offered, leads, feedback, greetings, memory, Arabic, French, prompt
injection, planning); a *core* subset of 12 for the sweeps. A case passes when the agent called the required tools
with the right arguments, avoided forbidden ones, and its answer contains the required facts and none of the
forbidden ones (it is judged on what the agent **did**, not on wording). Every run is one row in
`experiments/results/<plan>.jsonl` with the settings requested, sent and dropped, the full trace and the metrics; run
ids are `plan:config:case:repeat`. Columns below: `success` = share of cases passed, `format_errors` = corrections per
turn, `requests` = total requests, `distinct_2` = lexical variety (unique word pairs / all word pairs).

**Models:** `gemini-3.5-flash-lite` (the agent; it ignores temperature, top-p and top-k and rejects the penalties) and
`qwen3:4b-instruct` through Ollama (local; honours every setting). **Noise:** on the 12-case core set one case is 8
points; a control whose request equals the baseline (`s_flash:temp-0.0-control`, temperature is not sent) scored 12/12
against the baseline's 11/12, so one-case differences are noise.

### 7.1 Personas on each model (`p_flash`, `p_qwen`; full set, 34 cases)

In [12]:
show("personas_by_model")

,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
milo-flash,34,0.941,0.976,0.26,2.41,100,12.68,31.6,0.771
nora-flash,34,0.853,0.985,0.53,2.65,118,3.60,27.2,0.761
sami-flash,34,0.882,0.985,0.15,2.18,78,2.10,57.6,0.780
milo-qwen,34,0.618,0.882,0.24,2.24,76,14.31,23.4,0.706
nora-qwen,34,0.618,0.868,0.21,2.24,76,12.02,21.9,0.681
sami-qwen,34,0.588,0.926,0.18,2.09,72,17.58,51.4,0.773


### 7.2 Model type on the same 12 core cases (`model_type_core`)

In [13]:
show("model_type_core")

,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
milo-flash,12,0.917,1.0,0.25,2.42,36,12.61,31.1,0.876
nora-flash,12,1.000,1.0,0.42,2.67,39,3.60,31.7,0.819
sami-flash,12,0.917,1.0,0.17,2.33,30,2.12,56.6,0.858
milo-qwen,12,0.833,1.0,0.17,2.08,25,14.08,23.0,0.866
nora-qwen,12,0.833,1.0,0.08,2.17,26,13.09,21.7,0.848
sami-qwen,12,0.667,1.0,0.08,2.00,24,17.96,50.7,0.869


### 7.3 One setting at a time, on the model that honours it

`s_qwen`: from Milo's settings (temperature 0.7, top-p 0.9, top-k 40, frequency 0.2, presence 0.3, max_tokens 400),
on qwen, where every setting is honoured. `s_flash`: max_tokens and reasoning effort on Flash-Lite.

In [14]:
show("s_qwen")
show("s_flash")

,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
base,12,0.667,1.0,0.25,2.17,26,12.15,21.3,0.886
temp-0.0,12,0.833,1.0,0.17,2.08,25,11.84,23.9,0.892
temp-1.4,12,0.583,1.0,0.25,2.17,26,11.78,23.2,0.893
top_p-0.3,12,0.833,1.0,0.17,2.08,25,12.51,23.9,0.892
top_p-1.0,12,0.667,1.0,0.25,2.17,26,11.92,22.3,0.887
top_k-1,12,0.833,1.0,0.17,2.08,25,11.76,23.9,0.892
freq-0.0,12,0.667,1.0,0.17,2.08,25,11.37,22.5,0.914
freq-1.0,12,0.667,1.0,0.25,2.17,26,13.01,27.3,0.865
pres-0.0,12,0.667,1.0,0.25,2.17,26,11.82,22.2,0.888


,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
base,12,0.917,1.0,0.33,2.50,34,2.60,29.8,0.858
max_tokens-150,12,0.917,1.0,0.33,2.42,34,2.80,29.2,0.897
max_tokens-600,12,1.000,1.0,0.25,2.42,34,4.50,29.5,0.885
reasoning-minimal,12,1.000,1.0,0.42,2.50,36,6.50,32.0,0.866
reasoning-high,12,0.417,1.0,0.75,2.75,38,13.14,18.3,0.925
temp-0.0-control,12,1.000,1.0,0.17,2.33,32,12.22,32.2,0.878
reasoning-high-1200,12,1.000,1.0,0.25,2.42,33,15.26,35.2,0.852


### 7.4 Temperature and variety (`v_qwen`: Sami, 3 planning cases × 3 repeats)

In [15]:
show("v_qwen")

,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2,distinct_2_rep
config_id,,,,,,,,,,
sami-temp-0.0,9,0,1.0,0,1,9,26.91,145.7,0.290,0.309
sami-temp-1.4,9,0,1.0,0,1,9,30.81,153.1,0.789,0.835


### 7.5 Prompt engineering (`prompt_flash`: Milo on Flash-Lite, core; `prompt_confirm`: full set)

In [16]:
show("prompt_flash")
show("prompt_confirm")

,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
default,12,1.000,1.0,0.08,2.17,30,13.62,29.4,0.868
no-cot,12,0.833,1.0,0.00,1.83,22,10.15,33.7,0.839
zero-shot,12,1.000,1.0,0.33,2.42,36,12.38,31.9,0.887
few-shot,12,1.000,1.0,0.33,2.42,33,17.25,27.6,0.919
in-message,12,1.000,1.0,0.08,2.25,29,11.93,34.8,0.888


,runs,success,tool_precision,format_errors,steps,requests,latency_s,words,distinct_2
config_id,,,,,,,,,
default-full,34,0.912,0.971,0.32,2.44,98,13.46,30.0,0.754
in-message-full,34,0.941,0.971,0.03,2.12,75,9.31,31.6,0.724


### 7.6 Human ratings

`experiments/ratings/ratings_sample_flash.csv` holds 30 answers (10 cases × 3 personas on Flash-Lite) with blank 1–5
columns for helpfulness and persona fit, rated by hand.

In [17]:
ratings = pd.read_csv(ROOT / "experiments/ratings/ratings_sample_flash.csv")
rated = ratings.dropna(subset=["helpfulness_1to5", "persona_fit_1to5"])
if rated.empty:
    print(f"Not rated yet: 0 of {len(ratings)} answers have ratings.")
else:
    display(rated.groupby("persona")[["helpfulness_1to5", "persona_fit_1to5"]].agg(["mean", "count"]))

helpfulness_1to5       persona_fit_1to5      
                    mean count             mean count
persona                                              
milo                 4.6    10              4.7    10
nora                 4.4    10              4.9    10
sami                 4.5    10              3.9    10

## 8. Reflection

**Which persona gave the most helpful or natural results?**
On the shipped model, **Milo**: 32/34 cases (`p_flash:milo-flash`; 33/34 in fact, see the false negative in
section 9), with few format corrections (0.26 per turn).
Sami passed 30/34 and Nora 29/34; one to three cases apart, so the ranking is weak, but their weak spots are telling:
Nora, the most terse, needed the most format corrections (0.53 per turn) and planned poorly (1/3 planning cases);
Sami, the creative one, said "not offered" without logging the demand (1/3 not-offered cases). For their *own*
tasks each persona was the most natural: Nora gave exact numbers (the $150 difference and $74.50 / $149.50
deposits in the Stage 5 live run), Sami gave structured plans within the package limits (3/3 planning cases on
Flash-Lite). The voices were clearly different on both models: Sami's answers were about twice as long (58 and 51
words vs 27 to 32), and on qwen, where her temperature is actually applied, her wording was the most varied
(distinct-2 0.77 vs Nora's 0.68). **Human ratings** (7.6; 10 answers per persona, rated 1–5 by the author) agree:
Milo was the most helpful (4.6), Nora the most in character (persona fit 4.9) but once curt to the point of cold
(a thank-you answered in two flat sentences: 3/5), and Sami the least consistent (persona fit 3.9): her voice was
strong where her task was (the anniversary and graduation plans, 4–5) but she sounded like Nora on plain price
questions (2/5). So the best persona depends on the task, and a persona's voice is weakest outside its own focus.

**Which prompt and configuration combination performed best?**
**Milo on Flash-Lite with the shipped prompt**: visible Thought lines, one worked example, instructions in the system
prompt, low reasoning, max_tokens 400 (`prompt_confirm:default-full` 31/34, `p_flash:milo-flash` 32/34). Each prompt
factor was tested one at a time (`prompt_flash`): removing the Thought lines cost two cases (10/12, the model skipped
lookups), zero-shot and few-shot kept 12/12 but needed 4× the format corrections, and the placements tied. On the full
set, **putting the instructions in the user message** matched the success (32/34) with 1 format correction instead of
11 and 23% fewer requests (`prompt_confirm:in-message-full`): the better choice for cost; we kept the system prompt
because PDF 2.2 asks for it. Configuration: **reasoning effort and max_tokens must be chosen together**: high
reasoning at max_tokens 400 failed 7 of 12 cases because hidden thinking used up the budget (`s_flash:reasoning-high`
5/12), and passed 12/12 at 1200 (`s_flash:reasoning-high-1200`). Even Milo's *low* reasoning used 309 of his 400
tokens once and cut a reply mid-sentence (`p_flash:milo-flash:L1:0`, caught by the human rating), so 400 is too
tight: about 800 would be safer. On qwen the best sampling was **greedy**:
temperature 0.0, top-p 0.3 and top-k 1 produced *identical* runs (10/12 vs the baseline's 8/12), and temperature 1.4
was the worst (7/12, `s_qwen`).

**How well did the agent reason and use tools?**
On Flash-Lite, well: tool precision 0.97 to 0.99 (almost no unnecessary or wrong calls), a typical turn of 2.2 to 2.7
LLM calls (reason → act → reason → answer), lookups before every fact, the "log it first" rules followed on every
unanswerable question (3/3 for every persona), and the Arabic question answered in Arabic by all three (the French
one in French by Milo and Sami; Nora answered it in English). The weak point was the format: 0.15 to
0.53 corrections per turn, nearly all Flash-Lite's empty "malformed" replies (attempted native tool calls), which the
graph absorbs with a retry and a correction. On qwen, the reasoning was the problem, not the format: it often answered
from its own assumptions instead of looking things up (`missing tool` 6 to 8 times per persona), so it passed only 59
to 62% (`p_qwen`), wrote all 18 variance-plan answers without checking the package (`v_qwen` 0/18), and answered
the Arabic and French questions in English. Chain-of-thought mattered for grounding: without the Thought line even Flash-Lite skipped
lookups (`prompt_flash:no-cot`).

**Did the instructions block change the agent's behaviour in a noticeable way? How?**
Yes, in four measured ways. (1) The **worked example** stabilised the format: without it (`prompt_flash:zero-shot`)
format corrections quadrupled (0.33 vs 0.08 per turn). (2) Its **position**: with the output format in the middle,
5 of 6 identical requests came back empty; at the end, 1 of 6 (Stage 5). (3) Fixing a **persona vs instructions
conflict**: Milo's values ("people should feel heard before packages") beat the rule to ask for contact details, so
he asked about feelings instead of an email; after the values were reworded and rule 7 said "in that same reply", he
asked for name and email (Stage 5 live runs; `docs/evidence/stage5_live_traces*.jsonl`). (4) A rule that was too
strict: "never state a price no Observation contains" stopped Nora from giving deposit amounts until calculated
amounts were explicitly allowed. The A3 lessons ("**first** call record_feedback", "questions **about EchoNest**")
carried over: every Flash-Lite persona logged every unanswerable question.

**What were the biggest challenges?**
(1) **Settings that are silently ignored:** Gemini 3 Flash-Lite accepts temperature, top-p and top-k but ignores them,
and rejects the penalties; we measured each setting with a probe and ran the sampling experiments on a local model
that honours them. (2) **Text ReAct on a model trained for native tool calling:** Flash-Lite returned empty
"malformed" replies whenever the context looked like a tool call; it took a different scratchpad layout (Stage 4), the
output format last, a retry for empty replies and a format correction to make it reliable. Writing the parser, stop
sequence and recovery by hand (instead of A3's native tool calls) is also what makes the agent model-agnostic.
(3) **Quota and infrastructure:** 500 requests a day on the free tier (and a 429 whose "retry in 20 h" was wrong: it
reset at 10:00 Beirut), Gemma's free endpoint returning 503 for over an hour (dropped), and an 8 GB laptop on which
the default `qwen3:4b` tag turned out to be a thinking-only model. (4) **Small samples:** 12 cases make one case 8
points, so we measured the noise floor and report only differences that beat it.

## 9. Limitations and how to run

**Limitations.** One repeat per case for most plans, so small differences are noise. Two models (a hosted and a
local one); results may not transfer to larger models. The local model is slow (about 14 s per turn) and only on this
machine; its results are committed as evidence. Human ratings cover 30 answers on one model. Automatic scoring uses
fixed patterns, so it can miss a correct answer: one known false negative is `p_flash:milo-flash:A2:0`, where Milo
answered correctly in French ("entre 5 et 10 jours") but the frozen pattern only accepts "5 à 10" or "5-10"; the eval
set was not edited after the runs started. The agent answers only from EchoNest's documents and cannot see photos or
files.

**How to run** (from the repo root, Python 3.12):
```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements-dev.txt
cp .env.example .env          # add GEMINI_API_KEY; optional: LLM_PROVIDER, LLM_MODEL
python app.py                 # the Gradio assistant
for t in tests/test_*.py; do python "$t" > /dev/null || echo "FAILED: $t"; done
python scripts/run_experiments.py --plan experiments/plans/p_flash.json --dry-run
```
**Switching the model** is configuration only: set `LLM_PROVIDER` / `LLM_MODEL` in `.env` to an entry of
`config/models.json` (measure a new model once with `scripts/probe_settings.py`).